In [1]:
import sys
sys.path.append("src/")

from pathlib import Path
from proxyrecord import IpRecord
from utils import filecache

from geoip2.database import Reader
import folium
from folium.plugins import HeatMap


db_path = "./data/GeoLite2-City_20260421/GeoLite2-City.mmdb"
nl_center = [52.2, 5.3]

In [2]:
INPUT_FILE_PROXIES_BRIGHTDATA_SELFHOSTED = Path("data/prod-brightdata.json")
INPUT_FILE_PROXIES_BRIGHTDATA_IFCONFIG = Path("data/prod-withifconfig-brightdata.json")
INPUT_FILE_PROXIES_DECODO_SELFHOSTED = Path("data/prod-decodo.json")
INPUT_FILE_PROXIES_DECODO_IFCONFIG = Path("data/prod-withifconfig-decodo.json")

data_and_labels = [
    [INPUT_FILE_PROXIES_BRIGHTDATA_SELFHOSTED, "Brightdata to selfhosted infrastructure"],
    [INPUT_FILE_PROXIES_BRIGHTDATA_IFCONFIG, "Brightdata to ifconfig infrastructure"],
    [INPUT_FILE_PROXIES_DECODO_SELFHOSTED, "Decodo to selfhosted infrastructure"],
    [INPUT_FILE_PROXIES_DECODO_IFCONFIG, "Decodo to ifconfig infrastructure"]
]

In [3]:
results = dict()
for item in data_and_labels:
    [data, label] = item
    data_from_disk = filecache.read_list_from_file(data, IpRecord)
    
    for i in data_from_disk:
        results[i.ip] = i

In [4]:
def ip_to_latlon(reader, ip):
    try:
        resp = reader.city(ip)
        if resp.location.latitude is None or resp.location.longitude is None:
            return None
        return (resp.location.latitude, resp.location.longitude)
    except Exception:
        return None


In [5]:
def geolocate_ips(ip_list):
    coords = []
    with Reader(db_path) as reader:
        for ip in ip_list:
            loc = ip_to_latlon(reader, ip)
            if loc is not None:
                coords.append([loc[0], loc[1]])
    return coords

In [6]:
coords = geolocate_ips(results)
m = folium.Map(location=nl_center, zoom_start=7, tiles="OpenStreetMap")

HeatMap(
    coords,
    radius=15,
    blur=10,
    max_zoom=10,
).add_to(m)

m